# 手撕 KTO (Kahneman-Tversky Optimization)

## 背景
KTO 基于前景理论（prospect theory），不需要成对偏好数据，只需单点反馈（好/坏）。
对 desirable response 用收益函数，对 undesirable 用损失函数（损失更敏感）。
优势：数据需求量低，无需成对标注。

## 考察点
- 前景理论：人对损失比收益更敏感（λ > 1）
- KTO loss 公式
- 与 DPO 的区别（无需成对数据）

In [ ]:
import torch
import torch.nn.functional as F

def kto_loss(policy_logps, ref_logps, is_desirable: bool, beta: float = 0.1, lambda_u: float = 1.0) -> torch.Tensor:
    # policy_logps: log π(y|x) under policy
    # ref_logps: log π_ref(y|x) under reference
    # is_desirable: bool tensor, True=好回答, False=坏回答
    # beta: 温度, lambda_u: 损失厌恶系数
    rewards = beta * (policy_logps - ref_logps)  # 隐式奖励
    desirable = is_desirable.bool()
    # 收益函数（好回答）: σ(r) - σ(0) = σ(r) - 0.5
    # 损失函数（坏回答）: λ_u * (σ(0) - σ(r)) = λ_u * (0.5 - σ(r))
    sigma_r = torch.sigmoid(rewards)
    loss_desirable = (0.5 - sigma_r[desirable]).mean() if desirable.any() else torch.tensor(0.0)
    loss_undesirable = (lambda_u * (sigma_r[~desirable] - 0.5)).mean() if (~desirable).any() else torch.tensor(0.0)
    n_d, n_u = desirable.sum().item(), (~desirable).sum().item()
    total = n_d + n_u
    return (n_d / total) * loss_desirable + (n_u / total) * loss_undesirable

In [ ]:
# 验证 KTO loss
torch.manual_seed(42)
# 模拟 8 个样本：4 好 4 坏
policy_logps = torch.randn(8) * 2
ref_logps = torch.randn(8) * 2
is_desirable = torch.tensor([1, 1, 1, 1, 0, 0, 0, 0])
loss = kto_loss(policy_logps, ref_logps, is_desirable, beta=0.1)
assert loss.item() > 0 or loss.item() == 0, "loss 应非负"
# 验证：好回答 policy 概率上升时 loss 下降
good_logps = policy_logps.clone()
good_logps[:4] += 2.0  # 提高 desirable 的 policy logp
loss_better = kto_loss(good_logps, ref_logps, is_desirable, beta=0.1)
assert loss_better < loss, "提高好回答概率应降低 loss"
print(f"原始 loss: {loss.item():.6f}")
print(f"提升好回答后 loss: {loss_better.item():.6f}")
print("✅ KTO loss 验证通过")